# 11 — Retrain the two-stage TCN with hard negatives

This is a controlled follow-up to Notebook 08. The TCN architecture, 64-frame windows, group-separated split, optimizer, balanced samplers, early stopping, and validation-only threshold sweep stay the same. The deliberate change is the seventh landmark archive produced by Notebook 10:

- movement-like unsupported classes become stronger `unknown` evidence;
- supported variants expand bench press, bicep curl, deadlift, and tricep extension;
- the locked GainsByBrains video and labels remain absent from training.

Use a **T4 runtime**. All seven landmark archives download from the project branch; no upload is required. The output is `calorie_two_stage_tcn_hardneg_report.zip`.


In [ ]:
%pip install -q scikit-learn seaborn


In [ ]:
from pathlib import Path
import json
import random
import shutil
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from google.colab import files
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

SEED = 42
WINDOW = 64
STRIDE = 32
BATCH_SIZE = 64
MAX_EPOCHS = 40
PATIENCE = 8


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
assert DEVICE.type == "cuda", "Select a T4 runtime before training."


In [ ]:
DOWNLOAD_FROM_GITHUB = True
REPOSITORY_RAW = (
    "https://raw.githubusercontent.com/bariscelikk1/cALorie/"
    "feat/tcn-rebuild/ml/artifacts"
)
LANDMARK_ARCHIVES = [
    ("v1", "calorie_workoutfitness_landmarks_v1.zip"),
    ("v1", "calorie_mmfit_landmarks_v1.zip"),
    ("v1", "calorie_mmfit_unknown_v1.zip"),
    ("v2", "calorie_youtube_mixed_landmarks_v3.zip"),
    ("v3", "calorie_youtube_additional_landmarks_v1.zip"),
    ("v3", "calorie_youtube_training_landmarks_v4.zip"),
    ("v5", "calorie_workoutfitness_gate_hardneg_v1.zip"),
]

if DOWNLOAD_FROM_GITHUB:
    zip_names = []
    for version, name in LANDMARK_ARCHIVES:
        destination = Path("/content") / name
        url = f"{REPOSITORY_RAW}/{version}/landmarks/{name}"
        print("Downloading", name)
        request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request, timeout=180) as response, destination.open("wb") as output:
            shutil.copyfileobj(response, output)
        zip_names.append(str(destination))
        print(f"  {destination.stat().st_size / 1e6:.1f} MB")
else:
    print("Upload landmark ZIPs")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    assert zip_names, "Upload at least one landmark ZIP."

SOURCE_ROOT = Path("/content/two_stage_hardneg_sources")
if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir()
for index, name in enumerate(zip_names):
    destination = SOURCE_ROOT / f"source_{index}"
    destination.mkdir()
    with zipfile.ZipFile(name) as archive:
        archive.extractall(destination)
    print(Path(name).name, "->", destination)


In [ ]:
frames = []
for manifest_path in SOURCE_ROOT.rglob("manifest.csv"):
    frame = pd.read_csv(manifest_path).fillna("")
    frame["npz_path"] = frame["output_path"].map(lambda value: str(manifest_path.parent / value))
    frames.append(frame)

assert frames, "No manifest.csv was found."
manifest = pd.concat(frames, ignore_index=True)
manifest = manifest[
    (manifest["status"] == "ok")
    & manifest["npz_path"].map(lambda value: Path(value).exists())
].copy()
if "group_id" not in manifest or manifest["group_id"].eq("").any():
    manifest["group_id"] = manifest.apply(
        lambda row: f"{row['source_dataset']}:{row['clip_id']}", axis=1
    )

print("Valid clips:", len(manifest))
print("Groups:", manifest.group_id.nunique())
print(manifest.canonical_class.value_counts().sort_index())
assert "workoutfitness-gate-hardneg-v1" in set(manifest.source_dataset), (
    "Notebook 10 hard-negative archive was not loaded."
)


## Convert clips into windows

Each 64-frame window contains about 4.3 seconds at 15 FPS. The two models receive the same normalized 17-point x/y pose sequence. Every window retains its source `group_id`, so related windows cannot cross train, validation, and test boundaries.


In [ ]:
def resample_sequence(sequence, source_fps, target_fps=15):
    if not source_fps or abs(float(source_fps) - target_fps) < 0.1:
        return sequence
    duration = (len(sequence) - 1) / float(source_fps)
    target_length = max(2, int(round(duration * target_fps)) + 1)
    old_x = np.linspace(0, duration, len(sequence))
    new_x = np.linspace(0, duration, target_length)
    flat = sequence.reshape(len(sequence), -1)
    sampled = np.stack([
        np.interp(new_x, old_x, flat[:, index])
        for index in range(flat.shape[1])
    ], axis=1)
    return sampled.reshape(target_length, 17, 4).astype(np.float32)


windows, labels, groups, clip_ids, sources = [], [], [], [], []
for row in manifest.itertuples(index=False):
    with np.load(row.npz_path, allow_pickle=False) as item:
        sequence = item["landmarks"].astype(np.float32)
    if sequence.ndim != 3 or sequence.shape[1:] != (17, 4):
        print("Skipping unexpected shape:", row.npz_path, sequence.shape)
        continue
    sequence = resample_sequence(sequence, getattr(row, "source_fps", 15) or 15)
    starts = list(range(0, max(1, len(sequence) - WINDOW + 1), STRIDE))
    if len(sequence) > WINDOW and starts[-1] != len(sequence) - WINDOW:
        starts.append(len(sequence) - WINDOW)
    for start in starts:
        window = sequence[start:start + WINDOW]
        if len(window) < WINDOW:
            window = np.concatenate([
                window,
                np.repeat(window[-1:], WINDOW - len(window), axis=0),
            ])
        windows.append(window)
        labels.append(row.canonical_class)
        groups.append(row.group_id)
        clip_ids.append(row.clip_id)
        sources.append(row.source_dataset)

X = np.asarray(windows, dtype=np.float32)
labels = np.asarray(labels)
groups = np.asarray(groups)
clip_ids = np.asarray(clip_ids)
sources = np.asarray(sources)
ALL_CLASSES = sorted(np.unique(labels).tolist())
SUPPORTED_CLASSES = [name for name in ALL_CLASSES if name != "unknown"]
all_to_id = {name: index for index, name in enumerate(ALL_CLASSES)}
supported_to_id = {name: index for index, name in enumerate(SUPPORTED_CLASSES)}
y_all = np.asarray([all_to_id[name] for name in labels], dtype=np.int64)
y_gate = (labels != "unknown").astype(np.int64)  # 0 unknown, 1 supported
y_exercise = np.asarray([supported_to_id.get(name, -1) for name in labels], dtype=np.int64)

print("Windows:", X.shape)
print("Supported classes:", SUPPORTED_CLASSES)
print(pd.Series(labels).value_counts().sort_index())


In [ ]:
# One immutable group split is shared by the gate and exercise classifier.
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_val_idx, test_idx = next(outer.split(X, y_all, groups))
inner = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED + 1)
inner_train, inner_val = next(
    inner.split(X[train_val_idx], y_all[train_val_idx], groups[train_val_idx])
)
train_idx = train_val_idx[inner_train]
val_idx = train_val_idx[inner_val]

assert not (set(groups[train_idx]) & set(groups[val_idx]))
assert not (set(groups[train_idx]) & set(groups[test_idx]))
assert not (set(groups[val_idx]) & set(groups[test_idx]))

for split_name, indices in [("train", train_idx), ("validation", val_idx), ("test", test_idx)]:
    supported_present = sorted(set(labels[indices]) - {"unknown"})
    missing = sorted(set(SUPPORTED_CLASSES) - set(supported_present))
    print(
        split_name,
        "windows=", len(indices),
        "groups=", len(np.unique(groups[indices])),
        "gate counts=", np.bincount(y_gate[indices], minlength=2).tolist(),
        "missing supported classes=", missing,
    )


In [ ]:
class PoseWindowDataset(Dataset):
    def __init__(self, indices, targets, training=False):
        self.indices = np.asarray(indices)
        self.targets = targets
        self.training = training

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        actual = self.indices[item]
        sample = X[actual].copy()
        if self.training:
            sample[:, :, :3] += np.random.normal(
                0, 0.01, sample[:, :, :3].shape
            ).astype(np.float32)
        sample = sample[:, :, :2].reshape(WINDOW, -1).T
        return torch.from_numpy(sample), torch.tensor(self.targets[actual], dtype=torch.long)


def make_loaders(targets, train_indices, val_indices, test_indices):
    counts = np.bincount(targets[train_indices])
    weights = 1.0 / np.maximum(counts[targets[train_indices]], 1)
    sampler = WeightedRandomSampler(weights, len(weights), replacement=True)
    return (
        DataLoader(PoseWindowDataset(train_indices, targets, True), batch_size=BATCH_SIZE, sampler=sampler),
        DataLoader(PoseWindowDataset(val_indices, targets), batch_size=BATCH_SIZE, shuffle=False),
        DataLoader(PoseWindowDataset(test_indices, targets), batch_size=BATCH_SIZE, shuffle=False),
    )


gate_loaders = make_loaders(y_gate, train_idx, val_idx, test_idx)
exercise_train_idx = train_idx[y_exercise[train_idx] >= 0]
exercise_val_idx = val_idx[y_exercise[val_idx] >= 0]
exercise_test_idx = test_idx[y_exercise[test_idx] >= 0]
exercise_loaders = make_loaders(
    y_exercise, exercise_train_idx, exercise_val_idx, exercise_test_idx
)
print("Gate split sizes:", [len(loader.dataset) for loader in gate_loaders])
print("Exercise split sizes:", [len(loader.dataset) for loader in exercise_loaders])


In [ ]:
class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation, dropout=0.2):
        super().__init__()
        padding = dilation
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
        )
        self.residual = (
            nn.Conv1d(in_channels, out_channels, 1)
            if in_channels != out_channels else nn.Identity()
        )

    def forward(self, x):
        return torch.relu(self.net(x) + self.residual(x))


class SmallTCN(nn.Module):
    def __init__(self, input_features, num_classes):
        super().__init__()
        channels = [64, 96, 128, 128]
        blocks, current = [], input_features
        for level, output in enumerate(channels):
            blocks.append(TemporalBlock(current, output, dilation=2 ** level))
            current = output
        self.features = nn.Sequential(*blocks)
        self.classifier = nn.Linear(current, num_classes)

    def forward(self, x):
        return self.classifier(self.features(x).mean(dim=-1))


## Train both stages

Each model gets its own optimizer, early stopping, and balanced sampler. The gate checkpoint is selected by binary validation Macro F1. The exercise checkpoint is selected by 15-class validation Macro F1.


In [ ]:
def predict_model(model, loader):
    model.eval()
    truths, probabilities = [], []
    with torch.no_grad():
        for features, targets in loader:
            logits = model(features.to(DEVICE))
            truths.extend(targets.numpy())
            probabilities.extend(logits.softmax(1).cpu().numpy())
    probabilities = np.asarray(probabilities)
    return np.asarray(truths), probabilities.argmax(1), probabilities


def train_model(name, num_classes, loaders, seed):
    seed_everything(seed)
    train_loader, val_loader, _ = loaders
    model = SmallTCN(34, num_classes).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=3, factor=0.5
    )
    best_path = Path(f"/content/best_{name}.pt")
    best_f1, stale, history = -1.0, 0, []

    def run_epoch(loader, training):
        model.train(training)
        losses, truths, predictions = [], [], []
        context = torch.enable_grad() if training else torch.no_grad()
        with context:
            for features, targets in loader:
                features, targets = features.to(DEVICE), targets.to(DEVICE)
                if training:
                    optimizer.zero_grad(set_to_none=True)
                logits = model(features)
                loss = criterion(logits, targets)
                if training:
                    loss.backward()
                    optimizer.step()
                losses.append(loss.item() * len(targets))
                truths.extend(targets.cpu().numpy())
                predictions.extend(logits.argmax(1).cpu().numpy())
        return (
            sum(losses) / len(truths),
            f1_score(truths, predictions, average="macro", zero_division=0),
        )

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss, train_f1 = run_epoch(train_loader, True)
        val_loss, val_f1 = run_epoch(val_loader, False)
        scheduler.step(val_f1)
        history.append({
            "model": name, "epoch": epoch,
            "train_loss": train_loss, "train_macro_f1": train_f1,
            "val_loss": val_loss, "val_macro_f1": val_f1,
        })
        print(
            f"{name} {epoch:02d} train loss={train_loss:.4f} F1={train_f1:.3f} "
            f"| val loss={val_loss:.4f} F1={val_f1:.3f}"
        )
        if val_f1 > best_f1 + 1e-4:
            best_f1, stale = val_f1, 0
            torch.save(model.state_dict(), best_path)
        else:
            stale += 1
            if stale >= PATIENCE:
                print(name, "early stopping")
                break

    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    return model, history, best_f1


gate_model, gate_history, gate_best_f1 = train_model(
    "gate", 2, gate_loaders, SEED
)
exercise_model, exercise_history, exercise_best_f1 = train_model(
    "exercise", len(SUPPORTED_CLASSES), exercise_loaders, SEED + 1
)
print("Best validation F1 — gate:", round(gate_best_f1, 4))
print("Best validation F1 — exercise:", round(exercise_best_f1, 4))


## Calibrate the gate without touching the external test

For each validation window, the gate predicts a supported probability. We sweep thresholds from 0.05 to 0.95 and choose the value with the highest end-to-end validation Macro F1. This explicitly balances rejecting unknown motion against allowing supported exercises through.


In [ ]:
_, _, gate_val_probability = predict_model(gate_model, gate_loaders[1])

# The exercise model predicts every validation window for end-to-end calibration,
# including unknown windows that the gate may accidentally accept.
val_features = X[val_idx, :, :, :2].reshape(len(val_idx), WINDOW, -1).transpose(0, 2, 1)
exercise_val_probabilities = []
exercise_model.eval()
with torch.no_grad():
    for start in range(0, len(val_features), 256):
        batch = torch.from_numpy(val_features[start:start + 256]).to(DEVICE)
        exercise_val_probabilities.append(exercise_model(batch).softmax(1).cpu().numpy())
exercise_val_probabilities = np.concatenate(exercise_val_probabilities)
exercise_val_predictions = exercise_val_probabilities.argmax(1)

threshold_rows = []
for threshold in np.linspace(0.05, 0.95, 37):
    end_predictions = np.asarray([
        SUPPORTED_CLASSES[class_id] if supported_probability >= threshold else "unknown"
        for supported_probability, class_id in zip(
            gate_val_probability[:, 1], exercise_val_predictions
        )
    ])
    score = f1_score(
        labels[val_idx], end_predictions,
        labels=ALL_CLASSES, average="macro", zero_division=0,
    )
    supported_mask = labels[val_idx] != "unknown"
    unknown_mask = ~supported_mask
    threshold_rows.append({
        "threshold": float(threshold),
        "validation_macro_f1": float(score),
        "supported_accuracy": float((end_predictions[supported_mask] == labels[val_idx][supported_mask]).mean()),
        "unknown_rejection": float((end_predictions[unknown_mask] == "unknown").mean()),
    })

threshold_frame = pd.DataFrame(threshold_rows)
best_threshold_row = threshold_frame.sort_values(
    ["validation_macro_f1", "supported_accuracy"], ascending=False
).iloc[0]
GATE_THRESHOLD = float(best_threshold_row.threshold)
print("Chosen gate threshold:", GATE_THRESHOLD)
display(threshold_frame.sort_values("validation_macro_f1", ascending=False).head(10))


In [ ]:
def full_split_probabilities(indices):
    features = X[indices, :, :, :2].reshape(len(indices), WINDOW, -1).transpose(0, 2, 1)
    gate_parts, exercise_parts = [], []
    gate_model.eval()
    exercise_model.eval()
    with torch.no_grad():
        for start in range(0, len(features), 256):
            batch = torch.from_numpy(features[start:start + 256]).to(DEVICE)
            gate_parts.append(gate_model(batch).softmax(1).cpu().numpy())
            exercise_parts.append(exercise_model(batch).softmax(1).cpu().numpy())
    return np.concatenate(gate_parts), np.concatenate(exercise_parts)


gate_test_prob, exercise_test_prob = full_split_probabilities(test_idx)
gate_test_pred = gate_test_prob.argmax(1)
exercise_test_pred = exercise_test_prob.argmax(1)
end_to_end_pred = np.asarray([
    SUPPORTED_CLASSES[class_id] if supported_probability >= GATE_THRESHOLD else "unknown"
    for supported_probability, class_id in zip(gate_test_prob[:, 1], exercise_test_pred)
])

# Oracle classifier metrics use only truly supported test windows.
supported_test_mask = labels[test_idx] != "unknown"
classifier_truth = labels[test_idx][supported_test_mask]
classifier_pred = np.asarray([
    SUPPORTED_CLASSES[index] for index in exercise_test_pred[supported_test_mask]
])

metrics = {
    "experiment": "v5_hard_negative_controlled_retrain",
    "training_archives": [Path(name).name for name in zip_names],
    "seed": SEED,
    "window_frames": WINDOW,
    "stride_frames": STRIDE,
    "gate_threshold": GATE_THRESHOLD,
    "gate": {
        "accuracy": float(accuracy_score(y_gate[test_idx], gate_test_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_gate[test_idx], gate_test_pred)),
        "macro_f1": float(f1_score(y_gate[test_idx], gate_test_pred, average="macro", zero_division=0)),
    },
    "exercise_classifier_oracle": {
        "accuracy": float(accuracy_score(classifier_truth, classifier_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(classifier_truth, classifier_pred)),
        "macro_f1": float(f1_score(classifier_truth, classifier_pred, labels=SUPPORTED_CLASSES, average="macro", zero_division=0)),
    },
    "end_to_end": {
        "accuracy": float(accuracy_score(labels[test_idx], end_to_end_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(labels[test_idx], end_to_end_pred)),
        "macro_f1": float(f1_score(labels[test_idx], end_to_end_pred, labels=ALL_CLASSES, average="macro", zero_division=0)),
    },
    "classes": ALL_CLASSES,
    "supported_classes": SUPPORTED_CLASSES,
}
print(json.dumps(metrics, indent=2))


In [ ]:
REPORT_ROOT = Path("/content/calorie_two_stage_tcn_hardneg_report")
if REPORT_ROOT.exists():
    shutil.rmtree(REPORT_ROOT)
REPORT_ROOT.mkdir()

# Reports
pd.DataFrame(gate_history + exercise_history).to_csv(
    REPORT_ROOT / "training_history.csv", index=False
)
threshold_frame.to_csv(REPORT_ROOT / "gate_threshold_sweep.csv", index=False)
(REPORT_ROOT / "metrics.json").write_text(json.dumps(metrics, indent=2))

gate_report = classification_report(
    y_gate[test_idx], gate_test_pred,
    labels=[0, 1], target_names=["unknown", "supported"],
    output_dict=True, zero_division=0,
)
classifier_report = classification_report(
    classifier_truth, classifier_pred,
    labels=SUPPORTED_CLASSES, output_dict=True, zero_division=0,
)
end_report = classification_report(
    labels[test_idx], end_to_end_pred,
    labels=ALL_CLASSES, output_dict=True, zero_division=0,
)
(REPORT_ROOT / "gate_classification_report.json").write_text(json.dumps(gate_report, indent=2))
(REPORT_ROOT / "exercise_classification_report.json").write_text(json.dumps(classifier_report, indent=2))
(REPORT_ROOT / "end_to_end_classification_report.json").write_text(json.dumps(end_report, indent=2))

# Confusion matrices
plots = [
    ("gate", y_gate[test_idx], gate_test_pred, [0, 1], ["unknown", "supported"]),
    ("exercise", classifier_truth, classifier_pred, SUPPORTED_CLASSES, SUPPORTED_CLASSES),
    ("end_to_end", labels[test_idx], end_to_end_pred, ALL_CLASSES, ALL_CLASSES),
]
for name, truth, prediction, matrix_labels, tick_labels in plots:
    matrix = confusion_matrix(truth, prediction, labels=matrix_labels)
    plt.figure(figsize=(max(6, len(tick_labels) * 0.75), max(5, len(tick_labels) * 0.65)))
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", xticklabels=tick_labels, yticklabels=tick_labels)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(name.replace("_", " ").title() + " test confusion matrix")
    plt.tight_layout()
    plt.savefig(REPORT_ROOT / f"{name}_confusion_matrix.png", dpi=180)
    plt.show()

history_frame = pd.DataFrame(gate_history + exercise_history)
for name, group in history_frame.groupby("model"):
    group.plot(x="epoch", y=["train_macro_f1", "val_macro_f1"], figsize=(8, 4), grid=True, title=name)
    plt.tight_layout()
    plt.savefig(REPORT_ROOT / f"{name}_learning_curve.png", dpi=180)
    plt.show()

pd.DataFrame({
    "clip_id": clip_ids[test_idx],
    "source_dataset": sources[test_idx],
    "true_class": labels[test_idx],
    "gate_supported_probability": gate_test_prob[:, 1],
    "exercise_prediction": [SUPPORTED_CLASSES[index] for index in exercise_test_pred],
    "exercise_confidence": exercise_test_prob.max(axis=1),
    "predicted_class": end_to_end_pred,
}).to_csv(REPORT_ROOT / "test_window_predictions.csv", index=False)

torch.save({
    "gate_state_dict": gate_model.state_dict(),
    "exercise_state_dict": exercise_model.state_dict(),
    "classes": ALL_CLASSES,
    "supported_classes": SUPPORTED_CLASSES,
    "gate_threshold": GATE_THRESHOLD,
    "window_frames": WINDOW,
    "input_features": 34,
    "architecture": "two_stage_small_tcn_v2_hardneg",
    "experiment": "v5_hard_negative_controlled_retrain",
}, REPORT_ROOT / "two_stage_tcn.pt")

zip_path = shutil.make_archive(
    "/content/calorie_two_stage_tcn_hardneg_report", "zip", root_dir=REPORT_ROOT
)
print("Report ZIP:", zip_path)
files.download(zip_path)
